<a href="https://colab.research.google.com/github/ZixuanYang58/ZixuanYang58-DataScience-GenAI-Submissions/blob/main/Seminars/Week7_Seminar_Loan_Modelling_COMPLETED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning Modelling – Loan Data

**Week 7 Seminar | Option B**

Build and compare classification models using `loan_data.csv`. All features are numeric, with no missing values. The label is `loan_status`. Complete the tasks below in Google Colab.

**Student exercise:** Write your own code in the empty cells.

## Task 1 – Load and Inspect the Dataset

Upload `loan_data.csv` to Colab and load it into a dataframe. Inspect the first few rows, dataset dimensions, column types, and the distribution of `loan_status`. All columns are numeric and there are no missing values; verify these properties.

In [1]:
from google.colab import files
uploaded = files.upload()
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('loan_data.csv')
print(df.head())
print('Shape:', df.shape)
df.info()
print('Missing values:', df.isna().sum().sum())
print('Target counts:')
print(df['loan_status'].value_counts())

Saving loan_data.csv to loan_data.csv
   age  years_employed  annual_income  credit_score  credit_history_years  \
0   40            17.2          25579           692                   5.3   
1   33             7.3          43087           627                   3.5   
2   42             1.1          20840           689                   8.4   
3   53             0.5          29147           692                   9.8   
4   32            12.5          63657           630                   7.2   

   savings_assets  current_debt  defaults_on_file  delinquencies_last_2yrs  \
0             895         10820                 0                        0   
1             169         16550                 0                        1   
2              17          7852                 0                        0   
3            1480         11603                 0                        1   
4             209         12424                 0                        0   

   derogatory_marks  loan_amou

## Task 2 – Separate Predictors and Target

Set `loan_status` as the target (Y) and use the remaining columns as predictors (X). Check the dimensions of both.

In [2]:

X = df.drop(columns=['loan_status'])
y = df['loan_status']

print('X:', X.shape)
print('y:', y.shape)


X: (50000, 15)
y: (50000,)


## Task 3 – Split the Data

Create training and test sets, using 80% of the observations for training and 20% for testing. Preserve the class proportions and use a fixed random seed so your results are reproducible. Check the resulting shapes.

In [3]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=4567,
    stratify=y
)

print('Training:', X_train.shape, y_train.shape)
print('Test:', X_test.shape, y_test.shape)


Training: (40000, 15) (40000,)
Test: (10000, 15) (10000,)


## Task 4 – Scale the Predictors

Scale the predictors so they are on a comparable scale. Fit the scaling procedure using **training data only**, then apply the same transformation to the test data. Why should the test set not be used to fit the scaler?

In [4]:

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Scaled training shape:', X_train_scaled.shape)
print('Scaled test shape:', X_test_scaled.shape)


Scaled training shape: (40000, 15)
Scaled test shape: (10000, 15)


**Why is the scaler fitted on the training data only?**

The scaler is fitted on the training set only to prevent data leakage. The test set should remain unseen during model training and preprocessing so that it can provide a more reliable evaluation of the model's performance on new data.

## Task 5 – Train Machine Learning Models

Train and compare **logistic regression**, **random forest**, and **gradient boosting decision trees (GBDT)** with reasonable initial settings. Use the scaled predictors for logistic regression. Tree-based models do not require scaling, so you can use the unscaled predictors for them.

In [5]:

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        random_state=4567
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        random_state=4567,
        n_jobs=-1
    ),
    'GBDT': GradientBoostingClassifier(
        n_estimators=80,
        random_state=4567
    )
}

for name, model in models.items():
    training_features = (
        X_train_scaled
        if name == 'Logistic Regression'
        else X_train
    )

    model.fit(training_features, y_train)
    print(name, 'trained')


Logistic Regression trained
Random Forest trained
GBDT trained


## Task 6 – Evaluate and Compare Models

Evaluate each model on **both the training and test sets**. Report accuracy, macro precision, macro recall, and macro F1-score. Compare training and test performance. Which model generalizes best, and is there evidence of overfitting?

In [6]:

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

results = []

for name, model in models.items():
    for split, features, target in [
        (
            'Train',
            X_train_scaled if name == 'Logistic Regression' else X_train,
            y_train
        ),
        (
            'Test',
            X_test_scaled if name == 'Logistic Regression' else X_test,
            y_test
        )
    ]:
        pred = model.predict(features)

        precision, recall, f1, _ = precision_recall_fscore_support(
            target,
            pred,
            average='macro',
            zero_division=0
        )

        results.append({
            'Model': name,
            'Split': split,
            'Accuracy': accuracy_score(target, pred),
            'Macro Precision': precision,
            'Macro Recall': recall,
            'Macro F1': f1
        })

results_df = pd.DataFrame(results)
display(results_df.round(3))


,Model,Split,Accuracy,Macro Precision,Macro Recall,Macro F1
0,Logistic Regression,Train,0.838,0.837,0.835,0.836
1,Logistic Regression,Test,0.842,0.842,0.839,0.840
2,Random Forest,Train,1.000,1.000,1.000,1.000
3,Random Forest,Test,0.887,0.886,0.885,0.885
4,GBDT,Train,0.889,0.889,0.886,0.887
5,GBDT,Test,0.889,0.889,0.885,0.887


## Reflection

Which model would you choose based on the test results? What other checks would you want before using the model to support real lending decisions?

Three classification models were trained and evaluated: Logistic Regression, Random Forest, and Gradient Boosting Decision Trees (GBDT).

The models were compared using accuracy, macro precision, macro recall, and macro F1-score on both the training and test sets.

**GBDT achieved the highest test accuracy of 0.889 and the highest test macro F1-score of 0.887.** Random Forest achieved a slightly lower test macro F1-score of 0.885, while Logistic Regression obtained 0.840.

Random Forest showed evidence of overfitting, achieving a training macro F1-score of 1.000 compared with 0.885 on the test set. In contrast, GBDT demonstrated more consistent performance, with both training and test macro F1-scores of approximately 0.887.

**I would choose GBDT** as the preferred model for this exercise because it achieved the strongest test performance while showing relatively little evidence of overfitting.

Before using the model to support real lending decisions, I would examine the confusion matrix, particularly the consequences of incorrectly classifying high-risk applicants. I would also assess class imbalance, fairness across different applicant groups, probability calibration, and performance on new data.

Overall, GBDT provides the strongest results in this experiment, although further validation would be necessary before practical deployment.